# 贝叶斯优化

## 什么是贝叶斯优化？

贝叶斯优化是一种智能的优化方法，特别适合：
- 目标函数评估成本高
- 黑盒函数优化
- 全局优化问题

核心思想：
- 使用代理模型（高斯过程）建模目标函数
- 使用采集函数平衡探索和利用
- 迭代更新模型和选择下一个评估点

## 偏置引导的贝叶斯优化

在NSGABLACK框架中，**偏置系统**与贝叶斯优化形成强大的协同效应：

### 1. 偏置-贝叶斯协同机制
- **先验知识偏置**：将领域知识注入高斯过程先验
- **采集函数偏置**：修改采集函数以引导搜索方向
- **代理模型偏置**：构建有偏的代理模型加速收敛

### 2. 多层次偏置融合
- **数据层面偏置**：引导采样点生成
- **模型层面偏置**：影响高斯过程超参数
- **决策层面偏置**：调整采集函数权重

### 3. 动态偏置策略
- **早期阶段**：强偏置，快速定位有希望区域
- **中期阶段**：中等偏置，平衡探索与利用
- **后期阶段**：弱偏置，让贝叶斯优化主导精细搜索

### 4. 贝叶斯偏置的优势
- **样本效率**：减少昂贵的目标函数评估
- **知识融合**：将专家知识整合到优化过程
- **约束处理**：自然地处理约束条件
- **可解释性**：理解偏置对搜索的影响

In [ ]:
# 导入必要的库
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.spatial.distance import cdist
import warnings
warnings.filterwarnings('ignore')

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("环境准备完成")
print(f"NumPy 版本: {np.__version__}")

## 偏置增强的高斯过程代理模型

In [ ]:
# 导入必要的库
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.spatial.distance import cdist
from abc import ABC, abstractmethod
import warnings
warnings.filterwarnings('ignore')

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("环境准备完成")
print(f"NumPy 版本: {np.__version__}")

# ==================== 偏置系统基类 ====================
class BiasBase(ABC):
    """偏置系统基类"""
    
    def __init__(self, name: str, strength: float = 1.0):
        self.name = name
        self.strength = strength
        self.enabled = True
        self.history = []
        
    @abstractmethod
    def apply_bias(self, x: np.ndarray, bounds: List[Tuple[float, float]], 
                    context: Dict[str, Any] = None) -> np.ndarray:
        """应用偏置"""
        pass
    
    def update_context(self, new_best: Optional[np.ndarray] = None, 
                       improvement: Optional[float] = None):
        """更新偏置上下文"""
        if new_best is not None:
            self.history.append(new_best)
    
    def adapt_strength(self, iteration: int, max_iterations: int):
        """自适应调整偏置强度"""
        t = iteration / max_iterations
        # 贝叶斯优化中偏置衰减更快
        self.strength = np.exp(-2 * t)

# ==================== 贝叶斯优化专用偏置 ====================
class PriorKnowledgeBias(BiasBase):
    """先验知识偏置 - 注入领域知识到采样过程"""
    
    def __init__(self, strength: float = 1.0, preferred_regions: List[Dict] = None):
        super().__init__("先验知识偏置", strength)
        self.preferred_regions = preferred_regions or []
        
    def apply_bias(self, x: np.ndarray, bounds: List[Tuple[float, float]], 
                    context: Dict[str, Any] = None) -> np.ndarray:
        biased_x = x.copy()
        
        # 向偏好区域偏置
        for region in self.preferred_regions:
            if 'center' in region and 'attraction' in region:
                direction = region['center'] - x
                distance = np.linalg.norm(direction)
                if distance > 0:
                    # 吸引力随距离衰减
                    force = region['attraction'] * self.strength * np.exp(-distance / 2)
                    biased_x += force * direction / distance
        
        # 确保在边界内
        for i, (low, high) in enumerate(bounds):
            biased_x[i] = np.clip(biased_x[i], low, high)
        
        return biased_x

class AcquisitionBias(BiasBase):
    """采集函数偏置 - 修改采集函数行为"""
    
    def __init__(self, strength: float = 1.0, bias_type: str = 'exploration'):
        super().__init__("采集函数偏置", strength)
        self.bias_type = bias_type  # 'exploration', 'exploitation', 'constraint'
        
    def modify_acquisition(self, acq_values: np.ndarray, X: np.ndarray, 
                          context: Dict[str, Any] = None) -> np.ndarray:
        """修改采集函数值"""
        if self.bias_type == 'exploration':
            # 增强高不确定性区域的值
            if 'uncertainty' in context:
                uncertainty_bonus = self.strength * context['uncertainty']
                acq_values += uncertainty_bonus
                
        elif self.bias_type == 'exploitation':
            # 增强已知好区域的值
            if 'best_distance' in context:
                exploitation_bonus = self.strength * np.exp(-context['best_distance'])
                acq_values += exploitation_bonus
                
        elif self.bias_type == 'constraint':
            # 惩罚违反约束的区域
            if 'constraint_violation' in context:
                penalty = self.strength * context['constraint_violation']
                acq_values -= penalty
        
        return acq_values

class SurrogateModelBias(BiasBase):
    """代理模型偏置 - 影响高斯过程的行为"""
    
    def __init__(self, strength: float = 1.0, kernel_modification: str = 'adaptive'):
        super().__init__("代理模型偏置", strength)
        self.kernel_modification = kernel_modification
        
    def modify_kernel_parameters(self, base_alpha: float, base_length_scale: float,
                                iteration: int, max_iterations: int) -> Tuple[float, float]:
        """修改高斯过程核参数"""
        if self.kernel_modification == 'adaptive':
            # 自适应调整长度尺度
            t = iteration / max_iterations
            # 早期：大长度尺度（全局探索）
            # 后期：小长度尺度（局部精细）
            length_scale = base_length_scale * (2 - t)
            
            # 动态调整信号方差
            alpha = base_alpha * (1 + self.strength * (1 - t))
            
            return alpha, length_scale
        
        return base_alpha, base_length_scale

# ==================== 偏置增强的高斯过程 ====================
class BiasedGaussianProcess:
    """偏置增强的高斯过程模型"""
    
    def __init__(self, alpha=1.0, length_scale=1.0, bias: SurrogateModelBias = None):
        """初始化偏置增强的高斯过程"""
        self.base_alpha = alpha
        self.base_length_scale = length_scale
        self.bias = bias
        
        self.alpha = alpha
        self.length_scale = length_scale
        self.X_train = None
        self.y_train = None
        self.K_inv = None
        self.iteration = 0
        
    def rbf_kernel(self, X1, X2):
        """RBF核函数"""
        sqdist = cdist(X1, X2, 'sqeuclidean')
        return self.alpha * np.exp(-0.5 / self.length_scale**2 * sqdist)
    
    def fit(self, X, y):
        """拟合高斯过程"""
        self.X_train = X
        self.y_train = y
        
        # 应用偏置修改核参数
        if self.bias:
            self.alpha, self.length_scale = self.bias.modify_kernel_parameters(
                self.base_alpha, self.base_length_scale, 
                self.iteration, max_iterations=100  # 估计值
            )
        
        # 计算核矩阵
        K = self.rbf_kernel(X, X)
        K += 1e-8 * np.eye(len(X))
        
        # 计算逆矩阵
        self.K_inv = np.linalg.inv(K)
        self.iteration += 1
    
    def predict(self, X_test):
        """预测"""
        if self.X_train is None:
            return np.zeros(len(X_test)), np.ones(len(X_test))
        
        # K(X*, X)
        K_s = self.rbf_kernel(X_test, self.X_train)
        
        # K(X*, X*)
        K_ss = self.rbf_kernel(X_test, X_test)
        
        # 预测均值
        mean = K_s @ self.K_inv @ self.y_train
        
        # 预测方差
        var = K_ss - K_s @ self.K_inv @ K_s.T
        std = np.sqrt(np.diag(var))
        
        return mean.flatten(), std

# 测试偏置增强的高斯过程
print("测试偏置增强的高斯过程模型：")
print("-"*50)

# 创建测试数据
np.random.seed(42)
X_train = np.random.uniform(-3, 3, 5).reshape(-1, 1)
y_train = np.sin(X_train).flatten() + 0.1 * np.random.normal(0, 1, 5)

# 创建偏置
surrogate_bias = SurrogateModelBias(strength=0.5, kernel_modification='adaptive')

# 创建并拟合偏置增强的高斯过程
gp_biased = BiasedGaussianProcess(
    alpha=1.0, 
    length_scale=1.0, 
    bias=surrogate_bias
)
gp_biased.fit(X_train, y_train)

# 标准高斯过程对比
gp_standard = BiasedGaussianProcess(alpha=1.0, length_scale=1.0)
gp_standard.fit(X_train, y_train)

# 在测试点上预测
X_test = np.linspace(-4, 4, 100).reshape(-1, 1)
mean_biased, std_biased = gp_biased.predict(X_test)
mean_standard, std_standard = gp_standard.predict(X_test)

# 可视化对比
plt.figure(figsize=(15, 5))

# 标准高斯过程
plt.subplot(1, 3, 1)
plt.scatter(X_train, y_train, c='red', s=100, zorder=5, label='训练数据')
plt.plot(X_test, mean_standard, 'b-', linewidth=2, label='标准GP预测')
plt.fill_between(X_test.flatten(), mean_standard - 2*std_standard, 
                 mean_standard + 2*std_standard, alpha=0.3, color='blue')
plt.plot(X_test, np.sin(X_test), 'g--', linewidth=2, label='真实函数')
plt.xlabel('X')
plt.ylabel('Y')
plt.title('标准高斯过程')
plt.legend()
plt.grid(True, alpha=0.3)

# 偏置增强高斯过程
plt.subplot(1, 3, 2)
plt.scatter(X_train, y_train, c='red', s=100, zorder=5, label='训练数据')
plt.plot(X_test, mean_biased, 'r-', linewidth=2, label='偏置增强GP预测')
plt.fill_between(X_test.flatten(), mean_biased - 2*std_biased, 
                 mean_biased + 2*std_biased, alpha=0.3, color='red')
plt.plot(X_test, np.sin(X_test), 'g--', linewidth=2, label='真实函数')
plt.xlabel('X')
plt.ylabel('Y')
plt.title('偏置增强高斯过程')
plt.legend()
plt.grid(True, alpha=0.3)

# 对比
plt.subplot(1, 3, 3)
plt.plot(X_test, mean_standard, 'b-', linewidth=2, label='标准GP', alpha=0.7)
plt.plot(X_test, mean_biased, 'r-', linewidth=2, label='偏置增强GP', alpha=0.7)
plt.fill_between(X_test.flatten(), mean_standard - 2*std_standard, 
                 mean_standard + 2*std_standard, alpha=0.2, color='blue')
plt.fill_between(X_test.flatten(), mean_biased - 2*std_biased, 
                 mean_biased + 2*std_biased, alpha=0.2, color='red')
plt.xlabel('X')
plt.ylabel('Y')
plt.title('预测对比')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"偏置增强高斯过程参数:")
print(f"  信号方差: {gp_biased.alpha:.3f} (基准: {gp_biased.base_alpha:.3f})")
print(f"  长度尺度: {gp_biased.length_scale:.3f} (基准: {gp_biased.base_length_scale:.3f})")
print("\n偏置效果:")
print("- 早期阶段：较大的长度尺度，促进全局探索")
print("- 后期阶段：较小的长度尺度，实现局部精细搜索")

## 偏置增强的采集函数

In [ ]:
# 偏置增强的采集函数
class BiasedAcquisitionFunction:
    """偏置增强的采集函数基类"""
    
    def __init__(self, gp, xi=0.01, bias: AcquisitionBias = None):
        """初始化采集函数
        
        Args:
            gp: 高斯过程模型
            xi: 探索参数
            bias: 采集函数偏置
        """
        self.gp = gp
        self.xi = xi
        self.bias = bias
    
    def __call__(self, X):
        raise NotImplementedError

class BiasedExpectedImprovement(BiasedAcquisitionFunction):
    """偏置增强的期望改进（EI）采集函数"""
    
    def __call__(self, X):
        """计算偏置增强的期望改进"""
        # 预测
        mean, std = self.gp.predict(X)
        
        # 当前最佳目标值
        f_min = np.min(self.gp.y_train)
        
        # 计算标准EI
        with np.errstate(divide='warn'):
            imp = f_min - mean
            Z = imp / std
            ei = imp * norm.cdf(Z) + std * norm.pdf(Z)
            ei[std == 0.0] = 0.0
        
        # 应用偏置
        if self.bias:
            # 构建上下文
            context = {
                'uncertainty': std,
                'best_distance': np.linalg.norm(X - self.gp.X_train[np.argmin(self.gp.y_train)], axis=1),
                'mean': mean
            }
            ei = self.bias.modify_acquisition(ei, X, context)
        
        return ei

class BiasedUpperConfidenceBound(BiasedAcquisitionFunction):
    """偏置增强的上置信边界（UCB）采集函数"""
    
    def __init__(self, gp, kappa=2.576, xi=0.01, bias: AcquisitionBias = None):
        super().__init__(gp, xi, bias)
        self.kappa = kappa
    
    def __call__(self, X):
        """计算偏置增强的上置信边界"""
        # 预测
        mean, std = self.gp.predict(X)
        
        # 计算标准UCB
        ucb = mean + self.kappa * std
        
        # 应用偏置
        if self.bias:
            # 构建上下文
            context = {
                'uncertainty': std,
                'best_distance': np.linalg.norm(X - self.gp.X_train[np.argmin(self.gp.y_train)], axis=1),
                'mean': mean
            }
            ucb = self.bias.modify_acquisition(ucb, X, context)
        
        return ucb

class ProbabilityOfImprovement(BiasedAcquisitionFunction):
    """概率改进（PI）采集函数"""
    
    def __call__(self, X):
        """计算概率改进"""
        # 预测
        mean, std = self.gp.predict(X)
        
        # 当前最佳目标值
        f_min = np.min(self.gp.y_train)
        
        # 计算PI
        with np.errstate(divide='warn'):
            Z = (f_min - mean) / std
            pi = norm.cdf(Z)
            pi[std == 0.0] = 0.0
        
        # 应用偏置
        if self.bias:
            context = {
                'uncertainty': std,
                'best_distance': np.linalg.norm(X - self.gp.X_train[np.argmin(self.gp.y_train)], axis=1)
            }
            pi = self.bias.modify_acquisition(pi, X, context)
        
        return pi

# 测试偏置增强的采集函数
print("\n测试偏置增强的采集函数：")
print("-"*50)

# 使用之前的偏置增强高斯过程
exploration_bias = AcquisitionBias(strength=0.3, bias_type='exploration')
exploitation_bias = AcquisitionBias(strength=0.3, bias_type='exploitation')

# 创建偏置增强的采集函数
ei_biased = BiasedExpectedImprovement(gp_biased, bias=exploration_bias)
ei_exploit = BiasedExpectedImprovement(gp_biased, bias=exploitation_bias)
pi_biased = ProbabilityOfImprovement(gp_biased)
ucb_biased = BiasedUpperConfidenceBound(gp_biased, bias=exploration_bias)

# 在测试点上计算采集函数值
X_test = np.linspace(-4, 4, 100).reshape(-1, 1)
ei_values = ei_biased(X_test)
ei_exploit_values = ei_exploit(X_test)
pi_values = pi_biased(X_test)
ucb_values = ucb_biased(X_test)

# 可视化对比
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 高斯过程预测
ax1 = axes[0, 0]
ax1.scatter(gp_biased.X_train.flatten(), gp_biased.y_train, c='red', s=100, zorder=5)
mean_plot, std_plot = gp_biased.predict(X_test)
ax1.plot(X_test, mean_plot, 'b-', label='预测均值')
ax1.fill_between(X_test.flatten(), mean_plot - 2*std_plot, mean_plot + 2*std_plot, 
                 alpha=0.3, color='blue')
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_title('偏置增强高斯过程预测')
ax1.grid(True, alpha=0.3)

# EI-探索偏置
ax2 = axes[0, 1]
ax2.plot(X_test, ei_values, 'g-', linewidth=2, label='EI + 探索偏置')
next_point_idx = np.argmax(ei_values)
ax2.scatter(X_test[next_point_idx], ei_values[next_point_idx], 
           c='red', s=200, marker='*', label='下一个评估点', zorder=5)
ax2.set_xlabel('X')
ax2.set_ylabel('EI值')
ax2.set_title('期望改进 + 探索偏置')
ax2.legend()
ax2.grid(True, alpha=0.3)

# EI-利用偏置
ax3 = axes[0, 2]
ax3.plot(X_test, ei_exploit_values, 'b-', linewidth=2, label='EI + 利用偏置')
next_point_idx = np.argmax(ei_exploit_values)
ax3.scatter(X_test[next_point_idx], ei_exploit_values[next_point_idx], 
           c='red', s=200, marker='*', label='下一个评估点', zorder=5)
ax3.set_xlabel('X')
ax3.set_ylabel('EI值')
ax3.set_title('期望改进 + 利用偏置')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 概率改进
ax4 = axes[1, 0]
ax4.plot(X_test, pi_values, 'm-', linewidth=2, label='PI')
next_point_idx = np.argmax(pi_values)
ax4.scatter(X_test[next_point_idx], pi_values[next_point_idx], 
           c='red', s=200, marker='*', label='下一个评估点', zorder=5)
ax4.set_xlabel('X')
ax4.set_ylabel('PI值')
ax4.set_title('概率改进采集函数')
ax4.legend()
ax4.grid(True, alpha=0.3)

# UCB
ax5 = axes[1, 1]
ax5.plot(X_test, ucb_values, 'c-', linewidth=2, label='UCB + 探索偏置')
next_point_idx = np.argmax(ucb_values)
ax5.scatter(X_test[next_point_idx], ucb_values[next_point_idx], 
           c='red', s=200, marker='*', label='下一个评估点', zorder=5)
ax5.set_xlabel('X')
ax5.set_ylabel('UCB值')
ax5.set_title('上置信边界 + 探索偏置')
ax5.legend()
ax5.grid(True, alpha=0.3)

# 采集函数对比
ax6 = axes[1, 2]
ax6.plot(X_test, ei_values, 'g-', label='EI+探索', alpha=0.7)
ax6.plot(X_test, ei_exploit_values, 'b-', label='EI+利用', alpha=0.7)
ax6.plot(X_test, pi_values, 'm-', label='PI', alpha=0.7)
ax6.plot(X_test, ucb_values, 'c-', label='UCB+探索', alpha=0.7)
ax6.set_xlabel('X')
ax6.set_ylabel('采集函数值')
ax6.set_title('采集函数对比')
ax6.legend()
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("偏置增强采集函数说明：")
print("- EI+探索：增强高不确定性区域的吸引力")
print("- EI+利用：增强已知好区域附近的搜索")
print("- PI：概率改进，保守策略")
print("- UCB+探索：增强边界区域的探索")

# 分析偏置影响
print("\n偏置对采样位置的影响：")
print("-"*40)
for name, values in [("EI+探索", ei_values), ("EI+利用", ei_exploit_values), 
                     ("PI", pi_values), ("UCB+探索", ucb_values)]:
    idx = np.argmax(values)
    print(f"{name}: 推荐采样点 x = {X_test[idx, 0]:.3f}")

## 偏置增强的贝叶斯优化器

In [ ]:
# 偏置增强的贝叶斯优化器
class BiasedBayesianOptimizer:
    """偏置增强的贝叶斯优化器"""
    
    def __init__(self, bounds, acquisition='ei', n_init=5, max_iter=50,
                 gp_alpha=1.0, gp_length_scale=1.0, biases: List[BiasBase] = None,
                 random_state=None):
        """初始化偏置增强的贝叶斯优化器
        
        Args:
            bounds: 变量边界 [(low1, high1), (low2, high2), ...]
            acquisition: 采集函数类型 ('ei', 'ucb', 'pi')
            n_init: 初始随机点数
            max_iter: 最大迭代次数
            gp_alpha: 高斯过程信号方差
            gp_length_scale: 高斯过程长度尺度
            biases: 偏置列表
            random_state: 随机种子
        """
        self.bounds = np.array(bounds)
        self.n_var = len(bounds)
        self.n_init = n_init
        self.max_iter = max_iter
        self.biases = biases or []
        self.random_state = random_state
        
        if random_state is not None:
            np.random.seed(random_state)
        
        # 创建偏置组件
        surrogate_bias = next((b for b in self.biases if isinstance(b, SurrogateModelBias)), None)
        acquisition_bias = next((b for b in self.biases if isinstance(b, AcquisitionBias)), None)
        prior_bias = next((b for b in self.biases if isinstance(b, PriorKnowledgeBias)), None)
        
        # 初始化偏置增强的高斯过程
        self.gp = BiasedGaussianProcess(alpha=gp_alpha, length_scale=gp_length_scale, 
                                       bias=surrogate_bias)
        
        # 初始化偏置增强的采集函数
        if acquisition == 'ei':
            self.acquisition = BiasedExpectedImprovement(self.gp, bias=acquisition_bias)
        elif acquisition == 'ucb':
            self.acquisition = BiasedUpperConfidenceBound(self.gp, bias=acquisition_bias)
        elif acquisition == 'pi':
            self.acquisition = ProbabilityOfImprovement(self.gp, bias=acquisition_bias)
        else:
            raise ValueError("采集函数必须是 'ei', 'ucb' 或 'pi'")
        
        self.prior_bias = prior_bias
        
        # 存储历史
        self.X = []
        self.y = []
        self.best_x = None
        self.best_y = float('inf')
        
        # 性能统计
        self.bias_contributions = {bias.name: 0 for bias in self.biases}
        
        print(f"创建偏置增强贝叶斯优化器")
        print(f"  变量数: {self.n_var}")
        print(f"  边界: {self.bounds}")
        print(f"  初始点数: {n_init}")
        print(f"  最大迭代: {max_iter}")
        print(f"  采集函数: {acquisition}")
        print(f"  激活偏置: {[bias.name for bias in self.biases]}")
    
    def _random_sample(self, n_samples=1):
        """随机采样"""
        samples = np.random.uniform(
            self.bounds[:, 0],
            self.bounds[:, 1],
            (n_samples, self.n_var)
        )
        return samples
    
    def _biased_sample(self, n_samples=1):
        """偏置引导采样"""
        samples = []
        for _ in range(n_samples):
            # 生成基础随机采样点
            x = self._random_sample(1)[0]
            
            # 应用先验知识偏置
            if self.prior_bias:
                old_x = x.copy()
                x = self.prior_bias.apply_bias(x, self.bounds)
                if np.linalg.norm(x - old_x) > 1e-6:
                    self.bias_contributions[self.prior_bias.name] += 1
            
            samples.append(x)
        
        return np.array(samples)
    
    def _propose_location(self):
        """提议下一个评估位置"""
        # 生成候选点（混合随机和偏置采样）
        n_candidates = 1000
        n_biased = int(n_candidates * 0.3)  # 30%偏置采样
        
        candidates_random = self._random_sample(n_candidates - n_biased)
        candidates_biased = self._biased_sample(n_biased)
        
        candidates = np.vstack([candidates_random, candidates_biased])
        
        # 评估采集函数
        acq_values = self.acquisition(candidates)
        
        # 选择最佳候选
        best_idx = np.argmax(acq_values)
        best_candidate = candidates[best_idx]
        
        return best_candidate
    
    def optimize(self, objective_func):
        """执行优化"""
        print(f"\n开始偏置增强贝叶斯优化...")
        
        # 1. 初始随机采样（包含偏置）
        print(f"\n初始采样 ({self.n_init} 个点):")
        
        # 混合采样：50%随机，50%偏置
        n_random = self.n_init // 2
        n_biased = self.n_init - n_random
        
        X_init_random = self._random_sample(n_random)
        X_init_biased = self._biased_sample(n_biased)
        X_init = np.vstack([X_init_random, X_init_biased])
        
        for i, x in enumerate(X_init):
            y = objective_func(x)
            self.X.append(x)
            self.y.append(y)
            
            if y < self.best_y:
                self.best_y = y
                self.best_x = x.copy()
            
            sample_type = "偏置" if i >= n_random else "随机"
            print(f"  点 {i+1} ({sample_type}): x={x}, y={y:.4f}")
        
        # 2. 拟合初始高斯过程
        self.X = np.array(self.X)
        self.y = np.array(self.y)
        self.gp.fit(self.X, self.y)
        
        # 3. 迭代优化
        print(f"\n迭代优化 ({self.max_iter} 次迭代):")
        
        for iteration in range(self.max_iter):
            # 自适应调整偏置强度
            for bias in self.biases:
                bias.adapt_strength(iteration, self.max_iter)
            
            # 提议下一个位置
            x_next = self._propose_location()
            
            # 评估目标函数
            y_next = objective_func(x_next)
            
            # 更新数据
            self.X = np.vstack([self.X, x_next.reshape(1, -1)])
            self.y = np.append(self.y, y_next)
            
            # 更新最优解
            if y_next < self.best_y:
                improvement = self.best_y - y_next
                self.best_y = y_next
                self.best_x = x_next.copy()
                
                # 更新偏置上下文
                for bias in self.biases:
                    bias.update_context(self.best_x, improvement)
                
                print(f"  迭代 {iteration+1}: 改进! x={x_next}, y={y_next:.4f}")
            else:
                print(f"  迭代 {iteration+1}: x={x_next}, y={y_next:.4f}")
            
            # 重新拟合高斯过程
            self.gp.fit(self.X, self.y)
        
        print(f"\n优化完成！")
        print(f"最优值: {self.best_y:.4f}")
        print(f"最优解: {self.best_x}")
        
        # 输出偏置统计
        print("\n偏置贡献统计：")
        for bias_name, count in self.bias_contributions.items():
            percentage = count / (self.n_init + self.max_iter) * 100
            print(f"  {bias_name}: {count}次 ({percentage:.1f}%)")
        
        return self.best_x, self.best_y
    
    def plot_optimization(self, objective_func=None):
        """绘制优化过程（仅适用于1D问题）"""
        if self.n_var != 1:
            print("只能绘制1D问题的优化过程")
            return
        
        plt.figure(figsize=(15, 5))
        
        # 绘制目标函数（如果提供）
        if objective_func is not None:
            X_plot = np.linspace(self.bounds[0, 0], self.bounds[0, 1], 200)
            y_plot = [objective_func([x]) for x in X_plot]
            plt.subplot(1, 3, 1)
            plt.plot(X_plot, y_plot, 'g--', linewidth=2, label='目标函数')
            plt.scatter(self.X.flatten(), self.y, c='red', s=100, zorder=5, label='采样点')
            plt.scatter(self.best_x, self.best_y, c='blue', s=200, marker='*', zorder=6, label='最优解')
            plt.xlabel('X')
            plt.ylabel('Y')
            plt.title('优化过程')
            plt.legend()
            plt.grid(True, alpha=0.3)
        
        # 绘制高斯过程预测
        X_plot = np.linspace(self.bounds[0, 0], self.bounds[0, 1], 200).reshape(-1, 1)
        mean, std = self.gp.predict(X_plot)
        
        plt.subplot(1, 3, 2)
        plt.plot(X_plot, mean, 'b-', linewidth=2, label='GP预测')
        plt.fill_between(X_plot.flatten(), mean - 2*std, mean + 2*std, 
                         alpha=0.3, color='blue')
        plt.scatter(self.X.flatten(), self.y, c='red', s=100, zorder=5)
        plt.scatter(self.best_x, self.best_y, c='blue', s=200, marker='*', zorder=6)
        plt.xlabel('X')
        plt.ylabel('Y')
        plt.title('偏置增强高斯过程')
        plt.grid(True, alpha=0.3)
        
        # 绘制收敛历史
        plt.subplot(1, 3, 3)
        best_history = np.minimum.accumulate(self.y)
        plt.plot(range(len(best_history)), best_history, 'r-', linewidth=2)
        plt.xlabel('评估次数')
        plt.ylabel('当前最优值')
        plt.title('收敛历史')
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()

print("\n偏置增强贝叶斯优化器已定义")

## 偏置贝叶斯优化对比实验

In [ ]:
# 测试偏置增强的贝叶斯优化
print("测试偏置增强的贝叶斯优化算法：")
print("="*60)

# 定义测试函数
def test_function_1d(x):
    """1D测试函数 - 多峰函数"""
    x = x[0]
    return np.sin(x) + 0.1 * x + 0.05 * x**2

def hyperparameter_tuning(x):
    """机器学习超参数调优场景"""
    # x[0]: learning_rate (log scale)
    # x[1]: regularization_strength (log scale)
    lr = 10**x[0]  # 范围: 1e-5 到 1e-1
    reg = 10**x[1]  # 范围: 1e-4 到 1e2
    
    # 模拟验证损失（虚构的损失函数）
    loss = (lr - 0.01)**2 * 100 + (reg - 1.0)**2 * 0.1 + 0.05 * np.sin(10*lr) * np.cos(5*reg)
    return loss

# 创建业务场景配置
print("\n业务场景：机器学习超参数调优")
print("-"*40)
print("目标：优化学习率和正则化强度")
print("约束：学习率在[1e-5, 1e-1]，正则化在[1e-4, 1e2]")
print("挑战：评估成本高（需要训练模型）")

# 定义先验知识偏好区域
hyperparameter_prior = PriorKnowledgeBias(
    strength=0.7,
    preferred_regions=[
        {
            'center': np.array([-2.0, 0.0]),  # lr=0.01, reg=1.0
            'attraction': 0.8  # 强吸引力
        },
        {
            'center': np.array([-3.0, -1.0]),  # lr=0.001, reg=0.1
            'attraction': 0.4  # 中等吸引力
        }
    ]
)

# 对比实验：标准 vs 偏置增强贝叶斯优化
print("\n\n对比实验：标准贝叶斯优化 vs 偏置增强贝叶斯优化")
print("="*60)

# 1. 标准贝叶斯优化
print("\n1. 标准贝叶斯优化：")
print("-"*40)
bo_standard = BiasedBayesianOptimizer(
    bounds=[(-5, 1), (-4, 2)],  # log10(learning_rate), log10(regularization)
    acquisition='ei',
    n_init=6,
    max_iter=24,
    random_state=42,
    biases=[]
)

best_x_std, best_y_std = bo_standard.optimize(hyperparameter_tuning)

# 2. 偏置增强贝叶斯优化
print("\n\n2. 偏置增强贝叶斯优化：")
print("-"*40)

# 创建偏置组合
biases = [
    hyperparameter_prior,
    SurrogateModelBias(strength=0.5, kernel_modification='adaptive'),
    AcquisitionBias(strength=0.3, bias_type='exploitation')
]

bo_biased = BiasedBayesianOptimizer(
    bounds=[(-5, 1), (-4, 2)],
    acquisition='ei',
    n_init=6,
    max_iter=24,
    random_state=42,
    biases=biases
)

best_x_biased, best_y_biased = bo_biased.optimize(hyperparameter_tuning)

# 3. 不同偏置组合对比
print("\n\n3. 单一偏置效果对比：")
print("-"*40)

single_bias_results = {}
bias_configs = [
    ("仅先验偏置", [hyperparameter_prior]),
    ("仅代理偏置", [SurrogateModelBias(strength=0.5, kernel_modification='adaptive')]),
    ("仅采集偏置", [AcquisitionBias(strength=0.5, bias_type='exploration')])
]

for config_name, bias_list in bias_configs:
    print(f"\n{config_name}：")
    optimizer = BiasedBayesianOptimizer(
        bounds=[(-5, 1), (-4, 2)],
        acquisition='ei',
        n_init=6,
        max_iter=24,
        random_state=42,
        biases=bias_list
    )
    best_x, best_y = optimizer.optimize(hyperparameter_tuning)
    single_bias_results[config_name] = best_y

# 可视化对比结果
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. 收敛曲线对比
ax1 = axes[0, 0]
best_history_std = np.minimum.accumulate(bo_standard.y)
best_history_biased = np.minimum.accumulate(bo_biased.y)
ax1.plot(best_history_std, 'b-', label='标准贝叶斯优化', linewidth=2)
ax1.plot(best_history_biased, 'r-', label='偏置增强贝叶斯优化', linewidth=2)
ax1.set_xlabel('评估次数')
ax1.set_ylabel('最优目标值')
ax1.set_title('收敛曲线对比')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# 2. 最终结果对比
ax2 = axes[0, 1]
methods = ['标准贝叶斯', '偏置增强贝叶斯'] + list(single_bias_results.keys())
results = [best_y_std, best_y_biased] + list(single_bias_results.values())
colors = ['blue', 'red', 'green', 'orange', 'purple']

bars = ax2.bar(range(len(methods)), results, alpha=0.7, color=colors)
ax2.set_ylabel('最终目标值')
ax2.set_title('优化结果对比')
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_xticks(range(len(methods)))
ax2.set_xticklabels(methods, rotation=15, ha='right')

# 添加数值标签
for i, (bar, val) in enumerate(zip(bars, results)):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height(), 
             f'{val:.4f}', ha='center', va='bottom')

# 3. 参数空间探索（2D）
ax3 = axes[0, 2]
# 创建网格
lr_range = np.linspace(-5, 1, 50)
reg_range = np.linspace(-4, 2, 50)
LR, REG = np.meshgrid(lr_range, reg_range)
Z = np.zeros_like(LR)

# 计算网格上的函数值
for i in range(len(lr_range)):
    for j in range(len(reg_range)):
        Z[j, i] = hyperparameter_tuning([LR[j, i], REG[j, i]])

# 绘制等高线
contour = ax3.contour(LR, REG, Z, levels=20, alpha=0.5, cmap='viridis')
ax3.clabel(contour, inline=True, fontsize=8)

# 绘制采样点
ax3.scatter(bo_standard.X[:, 0], bo_standard.X[:, 1], 
           c='blue', s=50, alpha=0.7, label='标准采样')
ax3.scatter(bo_biased.X[:, 0], bo_biased.X[:, 1], 
           c='red', s=50, alpha=0.7, label='偏置采样')
ax3.scatter(best_x_std[0], best_x_std[1], 
           c='blue', s=200, marker='*', zorder=5)
ax3.scatter(best_x_biased[0], best_x_biased[1], 
           c='red', s=200, marker='*', zorder=5)

ax3.set_xlabel('log10(学习率)')
ax3.set_ylabel('log10(正则化强度)')
ax3.set_title('参数空间探索对比')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. 偏置贡献度
ax4 = axes[1, 0]
if bo_biased.bias_contributions:
    bias_names = list(bo_biased.bias_contributions.keys())
    contributions = list(bo_biased.bias_contributions.values())
    
    wedges, texts, autotexts = ax4.pie(contributions, labels=bias_names, 
                                         autopct='%1.1f%%', startangle=90)
    ax4.set_title('偏置贡献度分布')
    
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')

# 5. 偏置强度演化
ax5 = axes[1, 1]
iterations = range(30)  # 6初始 + 24迭代
for bias in biases:
    strengths = []
    for i in iterations:
        t = i / 30
        strength = np.exp(-2 * t)  # 贝叶斯优化特有的衰减
        strengths.append(strength)
    ax5.plot(iterations, strengths, label=bias.name, linewidth=2)

ax5.set_xlabel('评估次数')
ax5.set_ylabel('偏置强度')
ax5.set_title('偏置强度演化（贝叶斯优化特有）')
ax5.legend()
ax5.grid(True, alpha=0.3)

# 6. 高斯过程不确定性对比
ax6 = axes[1, 2]
# 在最优解周围计算不确定性
test_points = np.random.uniform(-5, 1, (100, 2))
test_points[:, 1] = np.random.uniform(-4, 2, 100)

_, std_std = bo_standard.gp.predict(test_points)
_, std_biased = bo_biased.gp.predict(test_points)

ax6.hist(std_std, bins=20, alpha=0.5, label='标准GP', color='blue', density=True)
ax6.hist(std_biased, bins=20, alpha=0.5, label='偏置GP', color='red', density=True)
ax6.set_xlabel('预测标准差')
ax6.set_ylabel('密度')
ax6.set_title('GP不确定性分布对比')
ax6.legend()
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 性能总结
print("\n\n性能总结：")
print("="*50)
improvement = (best_y_std - best_y_biased) / best_y_std * 100
print(f"偏置引导改进: {improvement:.2f}%")

print("\n最优超参数配置：")
print(f"\n标准贝叶斯优化：")
print(f"  学习率: {10**best_x_std[0]:.6f}")
print(f"  正则化强度: {10**best_x_std[1]:.6f}")
print(f"  损失: {best_y_std:.6f}")

print(f"\n偏置增强贝叶斯优化：")
print(f"  学习率: {10**best_x_biased[0]:.6f}")
print(f"  正则化强度: {10**best_x_biased[1]:.6f}")
print(f"  损失: {best_y_biased:.6f}")

print("\n偏置系统分析：")
print("- 先验知识偏置：引导搜索到经验上好的参数区域")
print("- 代理模型偏置：自适应调整核参数，早期探索后期精细")
print("- 采集函数偏置：增强利用倾向，加速收敛到已知好区域")
print("- 偏置衰减：在贝叶斯优化中快速衰减，让数据主导后期搜索")

# 业务价值分析
print("\n业务价值分析：")
print("-"*40)
print("1. 减少模型训练次数：", improvement > 0, f"（改进{improvement:.1f}%）")
print("2. 更好的泛化性能：找到接近理论最优的超参数")
print("3. 可解释性：偏置系统明确表达了对参数的先验知识")
print("4. 鲁棒性：混合了多种偏置策略，降低单一偏置的风险")

## 高级应用：贝叶斯偏置协同机制

In [ ]:
# 偏置协同的并行贝叶斯优化
class BiasedParallelBayesianOptimizer(BiasedBayesianOptimizer):
    """偏置协同的并行贝叶斯优化器"""
    
    def __init__(self, bounds, n_parallel=4, bias_coordination='competitive', **kwargs):
        """初始化并行贝叶斯优化器
        
        Args:
            bounds: 变量边界
            n_parallel: 并行度
            bias_coordination: 偏置协调策略 ('competitive', 'cooperative', 'diverse')
        """
        super().__init__(bounds, **kwargs)
        self.n_parallel = n_parallel
        self.bias_coordination = bias_coordination
        
        # 为每个并行进程创建不同的偏置配置
        self.parallel_biases = self._create_parallel_bias_configs()
        
        print(f"创建偏置协同并行贝叶斯优化器")
        print(f"  并行度: {n_parallel}")
        print(f"  偏置协调策略: {bias_coordination}")
    
    def _create_parallel_bias_configs(self):
        """创建并行的偏置配置"""
        configs = []
        
        if self.bias_coordination == 'competitive':
            # 竞争模式：每个进程专注一种偏置
            for i, bias in enumerate(self.biases[:self.n_parallel]):
                configs.append([bias])
                
        elif self.bias_coordination == 'cooperative':
            # 合作模式：所有偏置组合使用
            for i in range(self.n_parallel):
                configs.append(self.biases.copy())
                
        elif self.bias_coordination == 'diverse':
            # 多样化模式：混合不同偏置
            base_biases = self.biases.copy()
            for i in range(self.n_parallel):
                # 随机调整偏置强度
                config = []
                for bias in base_biases:
                    new_bias = bias.__class__(bias.name, bias.strength * np.random.uniform(0.5, 1.5))
                    config.append(new_bias)
                configs.append(config)
        
        return configs
    
    def optimize(self, objective_func):
        """执行偏置协同的并行优化"""
        print(f"\n开始偏置协同并行贝叶斯优化...")
        
        # 1. 初始采样（分散初始化）
        print(f"\n初始分散采样 ({self.n_init} 个点):")
        
        # 为每个并行进程分配不同的初始采样策略
        X_init = []
        y_init = []
        
        for i in range(self.n_init):
            # 轮换使用不同的偏置配置
            bias_idx = i % len(self.parallel_biases)
            temp_biases = self.parallel_biases[bias_idx]
            
            # 生成采样点
            if temp_biases and any(isinstance(b, PriorKnowledgeBias) for b in temp_biases):
                # 使用先验偏置
                prior_bias = next(b for b in temp_biases if isinstance(b, PriorKnowledgeBias))
                x = prior_bias.apply_bias(self._random_sample(1)[0], self.bounds)
            else:
                # 随机采样
                x = self._random_sample(1)[0]
            
            y = objective_func(x)
            
            X_init.append(x)
            y_init.append(y)
            
            if y < self.best_y:
                self.best_y = y
                self.best_x = x.copy()
            
            print(f"  点 {i+1}: x={x}, y={y:.4f}")
        
        # 2. 拟合初始模型
        self.X = np.array(X_init)
        self.y = np.array(y_init)
        self.gp.fit(self.X, self.y)
        
        # 3. 并行迭代优化
        print(f"\n并行优化 (每次评估 {self.n_parallel} 个点):")
        
        iteration = 0
        while len(self.X) < self.n_init + self.max_iter:
            batch_candidates = []
            batch_acq_values = []
            
            # 为每个并行位置生成候选
            for i in range(self.n_parallel):
                # 使用该位置的偏置配置
                temp_biases = self.parallel_biases[i % len(self.parallel_biases)]
                
                # 临时更新优化器的偏置
                old_biases = self.biases
                self.biases = temp_biases
                
                # 生成候选点
                x_candidate = self._propose_location()
                batch_candidates.append(x_candidate)
                
                # 恢复原始偏置
                self.biases = old_biases
            
            # 批量评估
            batch_results = []
            for x in batch_candidates:
                y = objective_func(x)
                batch_results.append((x, y))
                
                # 更新全局最优
                if y < self.best_y:
                    self.best_y = y
                    self.best_x = x.copy()
            
            # 更新数据
            for x, y in batch_results:
                self.X = np.vstack([self.X, x.reshape(1, -1)])
                self.y = np.append(self.y, y)
            
            # 重新拟合高斯过程
            self.gp.fit(self.X, self.y)
            
            iteration += 1
            batch_avg = np.mean([y for _, y in batch_results])
            print(f"  批次 {iteration}: 已评估 {len(self.X)} 个点, "
                  f"批次平均: {batch_avg:.4f}, 最优值: {self.best_y:.4f}")
        
        print(f"\n优化完成！")
        print(f"最优值: {self.best_y:.4f}")
        print(f"最优解: {self.best_x}")
        print(f"总评估次数: {len(self.y)}")
        
        return self.best_x, self.best_y

# 测试偏置协同并行贝叶斯优化
print("\n测试偏置协同的并行贝叶斯优化：")
print("-"*60)

# 对比不同的偏置协调策略
strategies = ['competitive', 'cooperative', 'diverse']
results = {}

for strategy in strategies:
    print(f"\n\n{strategy.upper()} 策略：")
    print("-"*40)
    
    bo_parallel = BiasedParallelBayesianOptimizer(
        bounds=[(-5, 1), (-4, 2)],
        acquisition='ei',
        n_init=6,
        max_iter=20,
        n_parallel=4,
        bias_coordination=strategy,
        random_state=42,
        biases=biases
    )
    
    best_x, best_y = bo_parallel.optimize(hyperparameter_tuning)
    results[strategy] = (best_x, best_y, bo_parallel)

# 可视化对比
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. 收敛曲线对比
ax1 = axes[0, 0]
for i, (strategy, (_, _, optimizer)) in enumerate(results.items()):
    best_history = np.minimum.accumulate(optimizer.y)
    ax1.plot(best_history, linewidth=2, label=f'{strategy}策略')
ax1.set_xlabel('评估次数')
ax1.set_ylabel('最优目标值')
ax1.set_title('不同偏置协调策略的收敛曲线')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# 2. 最终结果对比
ax2 = axes[0, 1]
strategy_names = list(results.keys())
final_values = [results[s][1] for s in strategy_names]
colors = ['blue', 'red', 'green']

bars = ax2.bar(strategy_names, final_values, alpha=0.7, color=colors)
ax2.set_ylabel('最终目标值')
ax2.set_title('偏置协调策略效果对比')
ax2.grid(True, alpha=0.3, axis='y')

# 添加数值标签
for bar, val in zip(bars, final_values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height(), 
             f'{val:.4f}', ha='center', va='bottom')

# 3. 采样分布对比（第一维）
ax3 = axes[1, 0]
for i, (strategy, (_, _, optimizer)) in enumerate(results.items()):
    ax3.scatter(optimizer.X[:, 0], optimizer.X[:, 1], 
               alpha=0.6, s=30, label=f'{strategy}')
ax3.set_xlabel('log10(学习率)')
ax3.set_ylabel('log10(正则化强度)')
ax3.set_title('参数空间采样分布对比')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. 策略分析
ax4 = axes[1, 1]
ax4.axis('off')
strategy_text = """
偏置协调策略分析：

COMPETITIVE（竞争）：
- 每个进程专注一种偏置
- 优点：深度探索每种偏置潜力
- 缺点：可能错过偏置组合效果

COOPERATIVE（合作）：
- 所有偏置协同工作
- 优点：充分利用所有偏置
- 缺点：可能过度拟合某些区域

DIVERSE（多样化）：
- 偏置强度随机化
- 优点：增加搜索多样性
- 缺点：随机性带来不确定性
"""
ax4.text(0.1, 0.9, strategy_text, transform=ax4.transAxes, 
         fontsize=10, verticalalignment='top', fontfamily='monospace')

plt.tight_layout()
plt.show()

# 最佳策略分析
best_strategy = min(results.keys(), key=lambda k: results[k][1])
print("\n\n偏置协同优化总结：")
print("="*50)
print(f"\n最佳策略: {best_strategy}")
print(f"最优值: {results[best_strategy][1]:.6f}")

print("\n各策略性能：")
for strategy in results:
    best_x, best_y, _ = results[strategy]
    print(f"  {strategy:12}: 损失={best_y:.6f}, "
          f"lr={10**best_x[0]:.6f}, reg={10**best_x[1]:.6f}")

print("\n偏置协同的核心价值：")
print("1. 多样性：不同偏置策略并行探索")
print("2. 鲁棒性：降低单一偏置失败的风险")
print("3. 效率：充分利用并行计算资源")
print("4. 自适应：策略本身可以动态调整")

## 贝叶斯优化与偏置系统总结

### 核心要点

1. **偏置-贝叶斯协同机制**
   - **先验知识偏置**：将专家知识注入高斯过程先验，引导初始搜索
   - **代理模型偏置**：自适应调整核参数，早期大范围探索，后期局部精细
   - **采集函数偏置**：修改采集函数行为，增强探索或利用倾向
   - **动态偏置策略**：偏置强度随时间衰减，让数据主导后期搜索

2. **贝叶斯优化中的偏置特色**
   - **样本效率**：偏置减少昂贵的目标函数评估次数
   - **知识融合**：自然地将领域知识整合到优化过程
   - **不确定性量化**：高斯过程提供的不确定性信息指导偏置调整
   - **自适应学习**：偏置根据模型预测动态调整

3. **多层次偏置融合**
   - **数据层面**：偏置引导初始采样点生成
   - **模型层面**：影响高斯过程超参数选择
   - **决策层面**：调整采集函数的权衡
   - **并行层面**：多种偏置策略协同探索

4. **实际应用价值**
   - **超参数调优**：利用经验快速找到好的参数配置
   - **实验设计**：在有偏置的搜索空间中高效采样
   - **昂贵优化**：减少仿真、物理实验等高成本评估
   - **约束处理**：通过偏置自然地避开不可行区域

### 偏置系统的核心优势

1. **智能引导**
   ```python
   # 先验知识快速定位好区域
   prior_bias = PriorKnowledgeBias(
       strength=0.7,
       preferred_regions=[
           {'center': [0.01, 1.0], 'attraction': 0.8}  # 经验最优
       ]
   )
   ```

2. **自适应探索**
   ```python
   # 代理模型偏置自适应调整
   surrogate_bias = SurrogateModelBias(
       strength=0.5,
       kernel_modification='adaptive'  # 早期探索，后期精细
   )
   ```

3. **动态平衡**
   ```python
   # 偏置强度随时间衰减
   def adapt_strength(self, iteration, max_iterations):
       t = iteration / max_iterations
       self.strength = np.exp(-2 * t)  # 贝叶斯优化中快速衰减
   ```

### 与其他算法的对比

| 特性 | 标准贝叶斯优化 | 偏置增强贝叶斯优化 | NSGABLACK框架特色 |
|------|---------------|-------------------|-----------------|
| 先验知识 | 隐式在先验中 | 显式偏置引导 | 多层次偏置系统 |
| 采样策略 | 纯数据驱动 | 偏置+数据混合 | 智能偏置组合 |
| 自适应性 | 核参数自适应 | 偏置强度自适应 | 全栈自适应 |
| 可解释性 | 黑盒 | 偏置效果可追踪 | 完全可解释 |
| 约束处理 | 惩罚函数 | 偏置回避 | 约束感知偏置 |

### 使用建议

1. **简单问题**
   - 使用先验知识偏置即可
   - 偏置强度设置为中等（0.5-0.7）

2. **复杂问题**
   - 组合多种偏置类型
   - 使用协同策略（competitive/cooperative）

3. **并行环境**
   - 利用并行贝叶斯优化
   - 选择合适的偏置协调策略

4. **实时场景**
   - 快速衰减偏置强度
   - 让数据主导优化过程

### 技术创新点

1. **偏置-代理协同**：偏置影响代理模型学习，代理模型指导偏置调整
2. **多层次融合**：从数据到决策的全链路偏置整合
3. **自适应策略**：偏置强度根据优化进程动态调整
4. **协同机制**：多种偏置策略在并行环境中协同工作

贝叶斯优化与偏置系统的结合，真正实现了"智能搜索"，将**专家经验**、**数据驱动**和**理论基础**完美融合，是NSGABLACK框架的核心创新之一！

通过偏置系统，贝叶斯优化不再是纯粹的黑盒搜索，而是变成了**可解释、可引导、可控**的智能优化过程。